# 03 · Evaluation

## Colab setup

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/UNIBO/Deep Learning/DL_3C")

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, str(PROJECT_DIR))
else:
    PROJECT_DIR = Path.cwd()

IMAGES_DIR = PROJECT_DIR / "images"
MODELS_DIR = PROJECT_DIR / "models"
IMAGES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print("Working folder:", Path.cwd())
print("Files:", os.listdir())

In [ ]:
import pickle
import random
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from neurosat import NeuroSAT, make_batch, to_device

sns.set_theme(style="whitegrid", context="notebook")

## Settings

In [ ]:
SEED = 42
DIM = 128
ROUNDS = 26
BATCH_SIZE = 32
ROUND_LIST = [4, 8, 16, 26, 48, 64, 96, 128]

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Load data and model

In [ ]:
with open("test.pkl", "rb") as f:
    test_data = pickle.load(f)

with open("test_large.pkl", "rb") as f:
    large_test_data = pickle.load(f)

model = NeuroSAT(DIM).to(device)
model.load_state_dict(torch.load(MODELS_DIR / "neurosat.pt", map_location=device))
model.eval()

print("Test:", len(test_data))
for n, data in large_test_data.items():
    print(f"Test n={n}:", len(data))

## Accuracy

In [ ]:
def predict(data, rounds):
    preds = []
    with torch.no_grad():
        for i in range(0, len(data), BATCH_SIZE):
            batch = to_device(make_batch(data[i:i + BATCH_SIZE]), device)
            logits = model(batch, rounds)
            preds += (logits > 0).long().tolist()
    return preds


def accuracy(data, rounds):
    preds = predict(data, rounds)
    labels = [p["label"] for p in data]

    correct = sum(p == l for p, l in zip(preds, labels))
    sat_correct = sum(p == l for p, l in zip(preds, labels) if l == 1)
    unsat_correct = sum(p == l for p, l in zip(preds, labels) if l == 0)

    return {
        "all": correct / len(labels),
        "sat": sat_correct / labels.count(1),
        "unsat": unsat_correct / labels.count(0),
    }

In [ ]:
test_acc = accuracy(test_data, ROUNDS)

print(f"Test accuracy:  {test_acc['all']:.3f}")
print(f"On SAT:         {test_acc['sat']:.3f}")
print(f"On UNSAT:       {test_acc['unsat']:.3f}")

## Accuracy vs rounds

In [ ]:
datasets = {"10–40": test_data}
for n, data in large_test_data.items():
    datasets[str(n)] = data

round_results = {}
for name, data in datasets.items():
    round_results[name] = []
    for rounds in ROUND_LIST:
        acc = accuracy(data, rounds)["all"]
        round_results[name].append(acc)
        print(f"n={name:6s} | rounds {rounds:3d} | acc {acc:.3f}")

In [ ]:
plt.figure(figsize=(7, 4))

for name, accs in round_results.items():
    sns.lineplot(x=ROUND_LIST, y=accs, marker="o", label=f"n = {name}")

plt.axvline(ROUNDS, color="gray", linestyle="--")
plt.xlabel("Message passing rounds")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "accuracy_vs_rounds.png", dpi=150)
plt.show()

## Decoding assignments

In [ ]:
def satisfies(clauses, assignment):
    for clause in clauses:
        if not any(assignment[abs(lit) - 1] == (lit > 0) for lit in clause):
            return False
    return True


def decode(problem, rounds):
    batch = to_device(make_batch([problem]), device)
    with torch.no_grad():
        logit, L_h = model(batch, rounds, return_literals=True)

    if logit.item() <= 0:
        return None

    clusters = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(L_h.cpu().numpy())
    n = problem["n_vars"]

    for true_cluster in [0, 1]:
        assignment = [clusters[i] == true_cluster for i in range(n)]
        if satisfies(problem["clauses"], assignment):
            return assignment

    return None


def decode_rate(data, rounds):
    sat_problems = [p for p in data if p["label"] == 1]
    solved = sum(decode(p, rounds) is not None for p in sat_problems)
    return solved / len(sat_problems)

In [ ]:
decode_results = {}
for name, data in datasets.items():
    decode_results[name] = {}
    for rounds in [ROUNDS, 128]:
        rate = decode_rate(data, rounds)
        decode_results[name][rounds] = rate
        print(f"n={name:6s} | rounds {rounds:3d} | solved {rate:.3f}")

## Literal embeddings

In [ ]:
for problem in test_data:
    if problem["label"] == 1:
        assignment = decode(problem, ROUNDS)
        if assignment is not None:
            break

batch = to_device(make_batch([problem]), device)
with torch.no_grad():
    _, L_h = model(batch, ROUNDS, return_literals=True)

points = PCA(n_components=2).fit_transform(L_h.cpu().numpy())
n = problem["n_vars"]

true_literals = [i if assignment[i] else n + i for i in range(n)]
false_literals = [n + i if assignment[i] else i for i in range(n)]
colors = sns.color_palette()

plt.figure(figsize=(6, 5))
sns.scatterplot(x=points[true_literals, 0], y=points[true_literals, 1], color=colors[0], s=60, label="True literals")
sns.scatterplot(x=points[false_literals, 0], y=points[false_literals, 1], color=colors[1], s=60, label="False literals")
plt.title(f"Literal embeddings, n = {n}")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "literal_embeddings.png", dpi=150)
plt.show()

## Summary

In [ ]:
print(f"{'n':>6s} | {'acc @26':>8s} | {'best acc':>8s} | {'solved @26':>10s} | {'solved @128':>11s}")
for name in datasets:
    acc_26 = round_results[name][ROUND_LIST.index(ROUNDS)]
    best = max(round_results[name])
    print(f"{name:>6s} | {acc_26:8.3f} | {best:8.3f} | {decode_results[name][ROUNDS]:10.3f} | {decode_results[name][128]:11.3f}")

In [ ]:
results = {
    "test_accuracy": test_acc,
    "round_list": ROUND_LIST,
    "accuracy_vs_rounds": round_results,
    "decode_rates": decode_results,
}

with open("results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved results.pkl, accuracy_vs_rounds.png and literal_embeddings.png")